# Project: Huấn luyện và export mô hình gợi ý nghề nghiệp CNTT

Notebook career được tổ chức theo đúng các mục của notebook **dự đoán giá xe**: đọc dữ liệu, xác định đặc trưng, xây dựng pipeline, đánh giá mô hình, giải thích đặc trưng, export, nạp lại để dự đoán và kiểm tra đầu vào.

Dữ liệu `data/students.csv` gồm điểm 12 môn học và nhóm nghề. Nhãn `derived` được suy ra từ chính điểm môn học, vì vậy các chỉ số chỉ phản ánh mức mô hình học lại quy tắc gán nhãn, **không phải độ chính xác dự đoán nghề nghiệp thực tế**. Tên sinh viên không phải đặc trưng và không hiển thị trong bảng xem trước.

In [ ]:
from pathlib import Path
import os
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, f1_score,
    precision_score, recall_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 110

PROJECT_ROOT = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / "career_ai").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find the project root containing career_ai/.")
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

from career_ai.domain_knowledge import (
    KEY_TO_SUBJECT, LABEL_DERIVED, LABEL_FEEDBACK, LABEL_SOURCE_COL,
    SUBJECTS, SUBJECT_KEYS,
)
from career_ai.model_calibration import (
    CALIBRATION_FOLDS, CALIBRATION_METHOD, build_calibrated_pipeline,
    evaluate_probability_calibration,
)
from career_ai.model_exporter import (
    export_model_package, load_model_with_metadata, validate_input_scores,
)
from career_ai.train import (
    build_models, next_model_version, print_feedback_validation,
    validate_on_feedback,
)

FEATURE_COLS = [SUBJECT_KEYS[subject] for subject in SUBJECTS]
TARGET = "career_group"
print(f"Project root: {PROJECT_ROOT}")

## 1. Đọc và khảo sát dữ liệu

Đọc điểm học phần từ `data/students.csv`. Bảng xem trước vẫn hiển thị dữ liệu, chỉ ẩn cột tên sinh viên.

In [1]:
data_path = PROJECT_ROOT / "data" / "students.csv"
df_all = pd.read_csv(data_path)
if LABEL_SOURCE_COL not in df_all.columns:
    df_all[LABEL_SOURCE_COL] = LABEL_DERIVED
df_all[LABEL_SOURCE_COL] = df_all[LABEL_SOURCE_COL].fillna(LABEL_DERIVED)

df_eda = df_all.loc[df_all[LABEL_SOURCE_COL] != LABEL_FEEDBACK].dropna(
    subset=FEATURE_COLS + [TARGET]
).reset_index(drop=True)
print(f"Dataset shape: {df_all.shape}")
print(f"Derived-label samples for evaluation: {len(df_eda)}")
print(f"Features: {len(FEATURE_COLS)} | Career groups: {df_eda[TARGET].nunique()}")
display(df_eda.drop(columns=["name"], errors="ignore").head())

Dataset shape: (356, 15)
Derived-label samples for evaluation: 355
Features: 12 | Career groups: 7


,sec,se,linux,db,graphics,network,analysis,image,java,csharp,web,ai,career_group,label_source
0,8.8,8.39,9.19,8.89,8.5,8.1,8.8,7.2,8.8,8.4,8.9,9.4,Kiểm thử & Quản lý dự án,derived
1,8.5,6.26,9.29,7.87,9.3,7.3,8.0,6.6,7.6,7.6,9.7,8.8,Phát triển Web,derived
2,7.5,6.45,7.78,6.75,9.0,6.7,3.0,6.8,5.9,7.7,5.9,5.9,Đồ họa & Game,derived
3,7.2,7.87,7.16,8.70,8.5,6.3,7.3,6.5,7.7,5.8,5.9,7.0,Kiểm thử & Quản lý dự án,derived
4,8.5,7.67,9.59,7.60,9.5,7.4,7.6,8.3,7.4,8.5,7.5,7.0,Đồ họa & Game,derived


In [ ]:
stats = df_eda[FEATURE_COLS].describe().T
stats.insert(0, "Subject", [KEY_TO_SUBJECT[key] for key in stats.index])
display(stats[["Subject", "mean", "std", "min", "50%", "max"]].round(2))

## 2. Xác định đặc trưng đầu vào

Mô hình dùng 12 cột điểm môn học (`sec` đến `ai`) làm đầu vào và `career_group` làm mục tiêu. Cột tên và nguồn nhãn không được đưa vào `X`. Chỉ nhãn `derived` dùng cho đánh giá; phản hồi người dùng (nếu có) chỉ được thêm ở lần fit cuối để export.

In [ ]:
df_eval = df_eda.copy()
df_final_train = df_all.dropna(subset=FEATURE_COLS + [TARGET]).reset_index(drop=True)
X = df_eval[FEATURE_COLS].astype(float)
y = df_eval[TARGET]

if y.value_counts().min() < 5:
    raise ValueError("Each career group needs at least 5 samples for 5-fold stratified CV.")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print(f"Train: {len(X_train)} | Test: {len(X_test)}")
print(f"Feedback rows reserved for final fit/export: {len(df_final_train) - len(df_eval)}")

## 3. Xây dựng Pipeline: tiền xử lý + model

Điểm số là dữ liệu số nên pipeline dùng `StandardScaler` trước bộ phân loại. Đặt scaler trong `Pipeline` để mỗi fold chỉ fit trên phần train của fold, tránh rò rỉ dữ liệu. So sánh Logistic Regression, Random Forest và Gradient Boosting; chọn thuật toán có F1 weighted 5-fold CV cao nhất.

In [ ]:
models = build_models()
comparison_rows = []
cv_scores = {}
for name, estimator in models.items():
    test_pipe = Pipeline([("preprocess", StandardScaler()), ("model", estimator)])
    test_pipe.fit(X_train, y_train)
    test_pred = test_pipe.predict(X_test)
    cv_pipe = Pipeline([("preprocess", StandardScaler()), ("model", estimator)])
    fold_scores = cross_val_score(cv_pipe, X, y, cv=skf, scoring="f1_weighted")
    cv_scores[name] = float(fold_scores.mean())
    comparison_rows.append({
        "Algorithm": name,
        "Accuracy": accuracy_score(y_test, test_pred),
        "Precision": precision_score(y_test, test_pred, average="weighted", zero_division=0),
        "Recall": recall_score(y_test, test_pred, average="weighted", zero_division=0),
        "F1 (Test)": f1_score(y_test, test_pred, average="weighted", zero_division=0),
        "F1 (5-Fold CV)": cv_scores[name],
    })
best_algo = max(cv_scores, key=cv_scores.get)
comparison = pd.DataFrame(comparison_rows)
comparison["Selected"] = comparison["Algorithm"].map(
    lambda name: "Deploy" if name == best_algo else ""
)
display(comparison.round(4))
print(f"Selected algorithm: {best_algo}")

## 4. Đánh giá model trên tập test

Fit pipeline có hiệu chuẩn xác suất trên tập Train, đánh giá trên Test chưa dùng để fit. Các chỉ số đo mức khớp với nhãn suy ra.

In [ ]:
best_pipe = build_calibrated_pipeline(models[best_algo])
best_pipe.fit(X_train, y_train)
y_pred = best_pipe.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average="weighted", zero_division=0)
recall = recall_score(y_test, y_pred, average="weighted", zero_division=0)
f1_test = f1_score(y_test, y_pred, average="weighted", zero_division=0)
print(f"Accuracy = {accuracy:.4f}")
print(f"Precision = {precision:.4f}")
print(f"Recall = {recall:.4f}")
print(f"F1 = {f1_test:.4f}")
print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))

labels = best_pipe.classes_
cm = confusion_matrix(y_test, y_pred, labels=labels)
display(pd.DataFrame(cm, index=labels, columns=labels))
fig, ax = plt.subplots(figsize=(8, 6))
image = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(labels)), labels, rotation=45, ha="right")
ax.set_yticks(range(len(labels)), labels)
for row in range(len(labels)):
    for col in range(len(labels)):
        ax.text(col, row, str(cm[row, col]), ha="center", va="center")
ax.set_xlabel("Predicted career group")
ax.set_ylabel("Derived-label career group")
ax.set_title("Test confusion matrix")
fig.colorbar(image, ax=ax)
plt.tight_layout()
plt.show()

print("\nReference validation using user feedback:")
print_feedback_validation(validate_on_feedback(best_pipe))

## 5. Mức độ quan trọng của các đặc trưng

Permutation importance đo mức thay đổi F1 khi xáo trộn điểm từng môn trên tập test. Đây là mức ảnh hưởng trong tập dữ liệu hiện tại, không phải quan hệ nhân quả.

In [ ]:
importance_result = permutation_importance(
    best_pipe, X_test, y_test, scoring="f1_weighted",
    n_repeats=10, random_state=42,
)
importance = pd.Series(
    importance_result.importances_mean,
    index=[KEY_TO_SUBJECT[key] for key in FEATURE_COLS],
).sort_values()
ax = importance.plot.barh(figsize=(9, 5), color="gray", edgecolor="black")
ax.set_xlabel("Mean decrease in weighted F1 after permutation")
ax.set_title("Subject feature importance")
plt.tight_layout()
plt.show()
display(importance.sort_values(ascending=False).to_frame("Permutation importance"))

## 6. Export model và metadata

Export pipeline (chuẩn hóa + phân loại + hiệu chuẩn) vào `models/current/model_pipeline.joblib`, kèm metadata về schema 12 đầu vào, khoảng điểm hợp lệ, chỉ số đánh giá và baseline. Artifact có phiên bản cũng được lưu trong `models/v{version}/`.

In [ ]:
calibration_metrics = evaluate_probability_calibration(models[best_algo], X, y, cv=skf)
calibrated_cv = cross_val_score(
    build_calibrated_pipeline(models[best_algo]), X, y,
    cv=skf, scoring="f1_weighted",
)
metrics_info = {
    "accuracy": accuracy,
    "precision": precision,
    "recall": recall,
    "f1_test": f1_test,
    "f1_cv_5fold_mean": float(calibrated_cv.mean()),
    "f1_cv_5fold_std": float(calibrated_cv.std()),
    **calibration_metrics,
    "calibration_method": CALIBRATION_METHOD,
    "calibration_cv_folds": CALIBRATION_FOLDS,
    "calibration_temperature": float(best_pipe.named_steps["model"].temperature_),
}

deployment_pipe = build_calibrated_pipeline(models[best_algo])
deployment_pipe.fit(df_final_train[FEATURE_COLS].astype(float), df_final_train[TARGET])
exported_meta = export_model_package(
    pipeline=deployment_pipe,
    algorithm_name=f"{best_algo} + temperature calibration",
    metrics=metrics_info,
    training_df=df_final_train,
    version=next_model_version(),
)
print("Model: models/current/model_pipeline.joblib")
print("Metadata: models/current/metadata.json")
print(f"Version: {exported_meta['model_version']}")
print(f"5-fold CV F1: {exported_meta['evaluation_metrics']['f1_cv_5fold_mean']:.4f}")

## 7. Kiểm tra: nạp lại model và dự đoán cho một hồ sơ mới

Nạp lại artifact đã export như API sẽ làm. Đầu vào được truyền dưới dạng DataFrame có đúng tên và thứ tự 12 cột điểm.

In [ ]:
loaded, loaded_metadata = load_model_with_metadata()
new_student = X_test.iloc[[0]][FEATURE_COLS].copy()
scores = new_student.iloc[0].to_dict()
is_valid, error_message, normalized_scores = validate_input_scores(scores)
if not is_valid:
    raise ValueError(error_message)
new_student = pd.DataFrame([normalized_scores], columns=FEATURE_COLS)
predicted_group = loaded.predict(new_student)[0]
probabilities = loaded.predict_proba(new_student)[0]
ranked_groups = sorted(
    zip(loaded.classes_, probabilities),
    key=lambda item: item[1],
    reverse=True,
)
print(f"Model version: {loaded_metadata['model_version']}")
print(f"Predicted career group: {predicted_group}")
display(pd.DataFrame(ranked_groups, columns=["Career group", "Model probability"]))

## 8. Một lỗi thường gặp khi đưa model vào ứng dụng

Điểm ngoài `[0, 10]` cần bị từ chối **trước khi** gọi model. API có thể dùng `validate_input_scores()` để kiểm tra dữ liệu đầu vào.

In [ ]:
invalid_scores = scores.copy()
invalid_scores[FEATURE_COLS[0]] = 10.5
is_valid, error_message, _ = validate_input_scores(invalid_scores)
print(f"Input is valid: {is_valid}")
print(f"Validation message: {error_message}")
assert not is_valid, "A score outside [0, 10] must be rejected." 